This notebook is for looking at the results of the tuning experiments.

In [1]:
# polars cant do hive partioning with json stuff
from pathlib import Path
from birdclef.spark import get_spark

scratch = Path("~/scratch/birdclef/2025").expanduser()

spark = get_spark()
tokenizer = (
    spark.read.json(
        f"{scratch}/mel2vec-tune/tokenizer/logistic/train",
        multiLine=True,
    )
    .where("_corrupt_record IS NULL")
    .drop("_corrupt_record")
    .orderBy("tokenizer_n_clusters")
)
tokenizer.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/16 00:49:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/06/16 00:49:30 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


+------------------+-------------------+------------------+---------+--------------------+-----------+------+-----------+------+------+
|          accuracy|     f1_macro_score|    f1_micro_score|tokenizer|tokenizer_n_clusters|vector_size|window|ns_exponent|sample|epochs|
+------------------+-------------------+------------------+---------+--------------------+-----------+------+-----------+------+------+
|0.5073630924988495|0.39936261337505424|0.5073630924988495|tokenizer|                4096|        256|    80|       0.75|1.0E-4|    20|
|0.5115048320294524| 0.4135536487234675|0.5115048320294524|tokenizer|                8192|        256|    80|       0.75|1.0E-4|    20|
|0.5036815462494247|0.42931196578782665|0.5036815462494247|tokenizer|               16383|        256|    80|       0.75|1.0E-4|    20|
|0.4995398067188219| 0.4283813903213475|0.4995398067188219|tokenizer|               32767|        256|    80|       0.75|1.0E-4|    20|
|0.4873446847676024| 0.3893013389879503|0.487344

In [2]:
tokenizer.select("tokenizer_n_clusters", "f1_macro_score", "f1_micro_score").show()

+--------------------+-------------------+------------------+
|tokenizer_n_clusters|     f1_macro_score|    f1_micro_score|
+--------------------+-------------------+------------------+
|                4096|0.39936261337505424|0.5073630924988495|
|                8192| 0.4135536487234675|0.5115048320294524|
|               16383|0.42931196578782665|0.5036815462494247|
|               32767| 0.4283813903213475|0.4995398067188219|
|               65535| 0.3893013389879503|0.4873446847676024|
+--------------------+-------------------+------------------+



In [7]:
tokenizer_timing = (
    spark.read.json(
        f"{scratch}/mel2vec-tune/tokenizer/word2vec",
        multiLine=True,
    )
    .where("_corrupt_record IS NULL")
    .drop("_corrupt_record")
)
tokenizer_timing.show()

25/06/16 01:48:22 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `epochs` already exists. Choose another name or rename the existing column. SQLSTATE: 42711


+------+-----------+------+------------------+-----------+------+-------+---------+--------------------+
|epochs|ns_exponent|sample|              time|vector_size|window|workers|tokenizer|tokenizer_n_clusters|
+------+-----------+------+------------------+-----------+------+-------+---------+--------------------+
|    20|       0.75|1.0E-4|1507.9780993629247|        256|    80|      8|tokenizer|               65535|
|    20|       0.75|1.0E-4|1046.1037343558855|        256|    80|      8|tokenizer|               16383|
|    20|       0.75|1.0E-4|1154.3622261900455|        256|    80|      8|tokenizer|               32767|
|    20|       0.75|1.0E-4| 890.5966026871465|        256|    80|      8|tokenizer|                4096|
|    20|       0.75|1.0E-4| 938.6446431339718|        256|    80|      8|tokenizer|                8192|
+------+-----------+------+------------------+-----------+------+-------+---------+--------------------+



In [8]:
w2v = (
    spark.read.json(
        f"{scratch}/mel2vec-tune/w2v/logistic/train",
        multiLine=True,
    )
    .where("_corrupt_record IS NULL")
    .drop("_corrupt_record")
    .orderBy("f1_macro_score", ascending=False)
)
w2v.show()

+-------------------+-------------------+-------------------+---------+--------------------+-----------+------+-----------+------+------+
|           accuracy|     f1_macro_score|     f1_micro_score|tokenizer|tokenizer_n_clusters|vector_size|window|ns_exponent|sample|epochs|
+-------------------+-------------------+-------------------+---------+--------------------+-----------+------+-----------+------+------+
| 0.5655775425678785|0.49467521152771055| 0.5655775425678785|tokenizer|               16383|       1028|    80|       0.75|1.0E-4|    20|
|  0.529222273354809|0.44877416277619336|  0.529222273354809|tokenizer|               16383|        384|    80|       0.75|1.0E-4|    20|
| 0.5446387482742752|0.44504761526763714| 0.5446387482742752|tokenizer|               16383|        512|    80|       0.75|1.0E-4|    20|
| 0.5059825126553152|0.43984339153384805| 0.5059825126553152|tokenizer|               16383|        256|    80|        0.0|1.0E-4|    20|
| 0.5108145421076852| 0.4271896635

In [9]:
w2v_timing = (
    spark.read.json(
        f"{scratch}/mel2vec-tune/w2v/word2vec",
        multiLine=True,
    )
    .where("_corrupt_record IS NULL")
    .drop("_corrupt_record")
)
w2v_timing.show()

25/06/16 01:48:30 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `epochs` already exists. Choose another name or rename the existing column. SQLSTATE: 42711


+------+-----------+------+------------------+-----------+------+-------+---------+--------------------+
|epochs|ns_exponent|sample|              time|vector_size|window|workers|tokenizer|tokenizer_n_clusters|
+------+-----------+------+------------------+-----------+------+-------+---------+--------------------+
|    20|       0.75|1.0E-4|1480.2317122467794|        256|   120|      8|tokenizer|               16383|
|    20|       -0.5|1.0E-4| 1470.953366188798|        256|   120|      8|tokenizer|               16383|
|    20|       0.75|1.0E-4|1673.5793089400977|        512|    80|      8|tokenizer|               16383|
|    20|       0.75|1.0E-4|1108.1109870010987|        256|    80|      8|tokenizer|               16383|
|    20|       0.75|1.0E-4| 569.5478072920814|        256|    40|      8|tokenizer|               16383|
|    20|       0.75|1.0E-4| 933.6023765578866|        128|    80|      8|tokenizer|               16383|
|    20|       -0.5|1.0E-4| 1042.778936198447|        2